The objective here will be to use a bitmap font atlas to match characters in the text field, and from that get an accurate OCR.

1. Load Atlas
2. Load Images
3. For each image:
   1. Prep a strip of the image for the date

In [7]:
import dask
import numpy as np
import pandas as pd
from numpy import ndarray
from dask.array.image import imread

# from dask.distributed import Client
# if 'client' not in locals():
#     client = Client()

filename_pattern = r'/home/rainybyte/AllSkyImages/2010-09/*.JPG'
images = imread(filename_pattern)
images

dask.array<imread, shape=(11696, 480, 640, 3), dtype=uint8, chunksize=(1, 480, 640, 3), chunktype=numpy.ndarray>

In [8]:
red = images.transpose(0, 3, 1, 2)[0][0]
red

dask.array<getitem, shape=(480, 640), dtype=uint8, chunksize=(480, 640), chunktype=numpy.ndarray>

Now that we have all the glyphs set up, we can classify the characters in the aliased-text images

In [9]:
from allsky.classifiers import char_glyphs, char_width, char_height, classify_at_cursor, classify_fields_block

In [10]:
from allsky.classifiers import digit_templates
digit_templates.shape

(11, 48)

In [11]:
best_char, best_score, scores = classify_at_cursor(
    image=red.astype(np.float32),
    x=5 + char_width * 1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=char_width,
    height=char_height,
)
best_char, best_score, scores

('0',
 0.9999374151229858,
 {'0': 0.9999374151229858,
  '1': -0.07989895343780518,
  '2': 0.4065017104148865,
  '3': 0.6845079660415649,
  '4': -0.08787599205970764,
  '5': 0.5073401927947998,
  '6': 0.7827118039131165,
  '7': -0.05048016086220741,
  '8': 0.783406138420105,
  '9': 0.7795872688293457,
  's': 0.20379485189914703})

Below we can see that this causes issues with the antialiased images.

In [12]:
images_2019 = imread(r'/home/rainybyte/AllSkyImages/2019-09/*.JPG')
red_2019 = images_2019.transpose(0, 3, 1, 2)[0][0]

best_char, best_score, scores = classify_at_cursor(
    image=red_2019.astype(np.float32),
    x=5 + char_width * 1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('5',
 0.36639389395713806,
 {'0': 0.33899861574172974,
  '1': -0.06114104762673378,
  '2': 0.2295859158039093,
  '3': 0.23315724730491638,
  '4': -0.165544331073761,
  '5': 0.36639389395713806,
  '6': 0.29352709650993347,
  '7': 0.020206689834594727,
  '8': 0.2833721339702606,
  '9': 0.25677579641342163,
  's': 0.02235008217394352})

In [13]:
from allsky.classifiers import (
    classify_date_string,
    classify_time_string,
    classify_exposure_string,
    classify_filename_string,
)

In [14]:
classify_date_string(red), classify_time_string(red), classify_exposure_string(red), classify_filename_string(red)

('2010/09/01', '00:58:36', '12.9028', '000029734')

In [15]:
batch = images.transpose(0, 3, 1, 2)[1000:1025]
batch_reds = batch[:,0]

In [16]:
batch_reds.map_blocks(classify_fields_block, dtype=pd.DataFrame, drop_axis=(1, 2)).compute()

,date,time,exposure,filename
0,2010/09/03,23:20:57,15.9205,000030909
0,2010/09/03,23:22:32,15.9205,000030910
0,2010/09/03,23:25:07,13.7774,000030911
0,2010/09/03,23:27:38,5.2167,000030912
0,2010/09/03,23:28:53,6.6013,000030913
0,2010/09/03,23:30:17,5.8054,000030914
0,2010/09/03,23:31:34,5.8054,000030915
0,2010/09/03,23:33:55,4.9670,000030916
0,2010/09/03,23:35:13,6.3169,000030917
0,2010/09/03,23:36:35,8.4513,000030918


In [17]:
import os
from glob import glob

image_folders = sorted(glob("/home/rainybyte/AllSkyImages/20*/"))
len(image_folders), image_folders

(100,
 ['/home/rainybyte/AllSkyImages/2010-08/',
  '/home/rainybyte/AllSkyImages/2010-09/',
  '/home/rainybyte/AllSkyImages/2010-10/',
  '/home/rainybyte/AllSkyImages/2010-11/',
  '/home/rainybyte/AllSkyImages/2010-12/',
  '/home/rainybyte/AllSkyImages/2011-01/',
  '/home/rainybyte/AllSkyImages/2011-02/',
  '/home/rainybyte/AllSkyImages/2011-03/',
  '/home/rainybyte/AllSkyImages/2011-04/',
  '/home/rainybyte/AllSkyImages/2011-05/',
  '/home/rainybyte/AllSkyImages/2011-06/',
  '/home/rainybyte/AllSkyImages/2011-07/',
  '/home/rainybyte/AllSkyImages/2011-08/',
  '/home/rainybyte/AllSkyImages/2011-09/',
  '/home/rainybyte/AllSkyImages/2011-10/',
  '/home/rainybyte/AllSkyImages/2011-11/',
  '/home/rainybyte/AllSkyImages/2011-12/',
  '/home/rainybyte/AllSkyImages/2012-01/',
  '/home/rainybyte/AllSkyImages/2012-02/',
  '/home/rainybyte/AllSkyImages/2012-03/',
  '/home/rainybyte/AllSkyImages/2012-04/',
  '/home/rainybyte/AllSkyImages/2012-05/',
  '/home/rainybyte/AllSkyImages/2012-06/',
  '/h

In [18]:
folder_set = set([os.path.basename(dir) for dir in glob("/home/rainybyte/AllSkyImages/20*")])
file_set = set([os.path.basename(file)[:-8] for file in glob("../../data/*.parquet")])
missing_set = set([])
missing_set = set([item for item in folder_set if item < "2016-02b"])
# missing_set = folder_set - file_set
missing_set = missing_set.difference({'2014-06'})
image_folders = [f"/home/rainybyte/AllSkyImages/{missing}/" for missing in sorted(list(missing_set))]
image_folders

[]

In [19]:
from allsky.processing import process_allsky_image_folder

tasks = [dask.delayed(process_allsky_image_folder)(folder, classify_fields_block) for folder in sorted(image_folders)]
dask.compute(tasks, scheduler='processes')
# for task in tasks:
#     try:
#         task.compute()
#     except OSError as e:
#         print(e.args, e.strerror, e)
#         continue
"All done!"

'All done!'